In [1]:
import pybamm
parameter_values = pybamm.ParameterValues(chemistry=pybamm.parameter_sets.Siegel2022)
spme = pybamm.lithium_ion.SPMe()
param = spme.param

In [2]:
def JS_Get_xy_from_voltage(parameter_values,initial_voltage):
    # First we solve for x_100 and y_100
    param = pybamm.LithiumIonParameters()
    Vmin = 2.7
    Vmax = 4.2
    Q_n = parameter_values.evaluate(param.n.cap_init)
    Q_p = parameter_values.evaluate(param.p.cap_init)
    Q_Li = parameter_values.evaluate(param.n_Li_particles_init)
    print(Q_n)
    print(Q_p)
    print(Q_Li)

    U_n = param.n.prim.U
    U_p = param.p.prim.U
    T_ref = param.T_ref

    model = pybamm.BaseModel()

    x_100 = pybamm.Variable("x_100")
    y_100 = (Q_Li - x_100 * Q_n) / Q_p

    y_100_min = 1e-10

    x_100_upper_limit = (Q_Li - y_100_min*Q_p)/Q_n
    print(x_100_upper_limit)
    model.algebraic = {x_100: U_p(y_100, T_ref) - U_n(x_100, T_ref) - Vmax}

    model.initial_conditions = {x_100: x_100_upper_limit}

    model.variables = {
        "x_100": x_100,
        "y_100": y_100
    }

    sim = pybamm.Simulation(model, parameter_values=parameter_values)
    sol = sim.solve([0])

    x_100 = sol["x_100"].data[0]
    y_100 = sol["y_100"].data[0]

    for var in ["x_100", "y_100"]:
        print(var, ":", sol[var].data[0])

    # Based on the calculated values for x_100 and y_100 we solve for x_0
    model = pybamm.BaseModel()

    x_0 = pybamm.Variable("x_0")
    Q = Q_n * (x_100 - x_0)
    y_0 = y_100 + Q/Q_p

    model.algebraic = {x_0: U_p(y_0, T_ref) - U_n(x_0, T_ref) - Vmin}
    model.initial_conditions = {x_0: 0.1}

    model.variables = {
        "Q": Q,
        "x_0": x_0,
        "y_0": y_0,
    }

    sim = pybamm.Simulation(model, parameter_values=parameter_values)
    sol = sim.solve([0])
    Q=sol["Q"].data[0]

    for var in ["Q", "x_0", "y_0"]:
        print(var, ":", sol[var].data[0])


    # Given x_100, y_100 and Q, solve for initial stored charge to get the right voltage
    Q_init = pybamm.Variable("Q_init")
    #Q = Q_n * (x_100 - x_0)
    #y_0 = y_100 + Q/Q_p

    model.algebraic = {Q_init: U_p(y_100-(Q_init-Q)/Q_p, T_ref) - U_n(x_100+(Q_init-Q)/Q_n, T_ref) - initial_voltage}
    model.initial_conditions = {Q_init: Q}

    model.variables = {
        "Q_init": Q_init,
    }

    sim = pybamm.Simulation(model, parameter_values=parameter_values)
    sol = sim.solve([0])


    for var in ["Q_init"]:
        print(var, ":", sol[var].data[0])
    Q_init=sol["Q_init"].data[0]
    return Q_init, x_100+(Q_init-Q)/Q_n, y_100-(Q_init-Q)/Q_p

In [3]:
parameter_values = pybamm.ParameterValues("Siegel2022")
parameter_values.update(
    {
        "Negative electrode active material volume fraction": 0.66735,
        "Positive electrode active material volume fraction": 0.70513,
    #    "Initial concentration in negative electrode [mol.m-3]":cs_n_init,
    #    "Initial concentration in positive electrode [mol.m-3]":cs_p_init,
        "Initial temperature [K]": 273.15+25,
        "Ambient temperature [K]": 273.15+25,
        "Lower voltage cut-off [V]": 2.7, 
        "Upper voltage cut-off [V]": 4.35
    },check_already_exists=False,)

q_init, x_init, y_init=JS_Get_xy_from_voltage(parameter_values,3.5)
print(q_init)
print(x_init)
print(y_init)

4.045800460710562
5.482783852438347
0.18246747934269072
0.045100464189073146


SolverError: CasADi solver failed because the following interpolation bounds were exceeded at the initial conditions: ['NMC_ocp_Siegel lower bound']. You may need to provide additional interpolation points outside these bounds.